# Fraud Mitigation Agent · 04 Rule Tools

Las reglas son configuración versionada, no lógica escondida dentro del prompt.


In [ ]:
import os, sys, json
from pathlib import Path

REPO_DIR = globals().get("REPO_DIR", "/content/fraud-mitigation-agent-workshop")
src = Path(REPO_DIR) / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from fraud_mitigation_agent.synthetic import seed_demo_data
from fraud_mitigation_agent.local import InMemoryDB

if "db" not in globals():
    from fraud_mitigation_agent.config import Settings
    from fraud_mitigation_agent.db import get_client, get_database
    settings = Settings.from_env()
    if settings.mongodb_uri:
        client = get_client(settings.mongodb_uri)
        db = get_database(client, settings.database_name)
    else:
        db = InMemoryDB()
seed_demo_data(db, reset=False)
print("Runtime listo:", type(db).__name__)


In [ ]:
from fraud_mitigation_agent.tools.transactions import get_transaction
from fraud_mitigation_agent.tools.customer_context import get_customer_state
from fraud_mitigation_agent.tools.rules import evaluate_rules, get_rules_config

tx = get_transaction(db, "tx-risky-001").data
customer = get_customer_state(db, tx["customer_id"]).data
print(json.dumps(get_rules_config(db).as_dict(), indent=2, default=str))
print(json.dumps(evaluate_rules(db, tx, customer).as_dict(), indent=2, default=str))
